# Case Study 2: Multivariate Time Series Forecasting — Jena Climate

## RNN vs LSTM vs GRU Comprehensive Comparison

---

### Objective
Predict temperature 24 hours ahead using 14 weather features recorded every 10 minutes at the Max Planck Institute in Jena, Germany.

### What You Will Learn
1. Multivariate input → univariate output forecasting
2. Handling large time series with subsampling
3. Multi-step forecasting at different horizons
4. Feature ablation: univariate vs multivariate LSTM
5. Persistence baseline comparison

### Dataset
- **Jena Climate**: 420,551 rows, 14 weather features, 10-minute intervals (2009–2016)
- We subsample to hourly (every 6th row) for tractable training

---
## 1. Environment Setup

In [ ]:
import random
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch: {torch.__version__} | Device: {DEVICE}')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')
COLORS = {'RNN': '#e74c3c', 'LSTM': '#2ecc71', 'GRU': '#3498db'}

---
## 2. Data Loading & EDA

In [ ]:
# Load the full dataset
df = pd.read_csv('Time_Series_Forecasting/jena_climate/jena_climate_2009_2016.csv')
print(f'Full dataset: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()

In [ ]:
# Parse datetime and subsample to hourly (every 6th row)
df['Date Time'] = pd.to_datetime(df['Date Time'], format='%d.%m.%Y %H:%M:%S')
df.set_index('Date Time', inplace=True)

# Subsample: 10-min → hourly
df = df.iloc[::6, :].copy()
print(f'After hourly subsampling: {df.shape}')
print(f'Date range: {df.index[0]} to {df.index[-1]}')
print(f'\nMissing values per column:\n{df.isnull().sum()}')

In [ ]:
df.describe().round(2)

In [ ]:
# Multi-feature time series plot
features = df.columns.tolist()
fig, axes = plt.subplots(len(features), 1, figsize=(16, 3*len(features)), sharex=True)

for ax, feat in zip(axes, features):
    ax.plot(df.index, df[feat], linewidth=0.3, color='#2c3e50')
    ax.set_ylabel(feat, fontsize=8)
    ax.tick_params(labelsize=7)

axes[0].set_title('Jena Climate — All 14 Features (Hourly)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(12, 10))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, ax=ax, annot_kws={'size': 7})
ax.set_title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

# Highlight correlations with temperature
temp_corr = corr['T (degC)'].drop('T (degC)').sort_values(ascending=False)
print('Correlations with Temperature T (degC):')
for feat, val in temp_corr.items():
    print(f'  {feat:25s}: {val:+.3f}')

In [ ]:
# Temperature zoom: 1 week
fig, axes = plt.subplots(2, 1, figsize=(16, 8))

# 1 year overview
year_data = df['2015']
axes[0].plot(year_data.index, year_data['T (degC)'], linewidth=0.5, color='#e74c3c')
axes[0].set_title('Temperature — Year 2015', fontsize=13)
axes[0].set_ylabel('T (degC)')

# 1 week zoom
week_data = df['2015-06-01':'2015-06-07']
axes[1].plot(week_data.index, week_data['T (degC)'], 'o-', linewidth=1, markersize=2, color='#e74c3c')
axes[1].set_title('Temperature — First Week of June 2015 (Daily Cycles Visible)', fontsize=13)
axes[1].set_ylabel('T (degC)')
axes[1].set_xlabel('Date')

plt.tight_layout()
plt.show()

---
## 3. Preprocessing

**Strategy:**
- Chronological split: Train (2009-2014), Validation (2015), Test (2016)
- StandardScaler fitted on training data only
- Input window: past 120 hours (5 days) → predict temperature at +24h
- All 14 features as input, temperature as single output

In [ ]:
# Target column
TARGET_COL = 'T (degC)'
target_idx = df.columns.tolist().index(TARGET_COL)
print(f'Target: {TARGET_COL} (column index {target_idx})')

# Chronological split by year
train_df = df[:'2014'].copy()
val_df = df['2015'].copy()
test_df = df['2016'].copy()

print(f'Train: {len(train_df)} samples ({train_df.index[0].year}-{train_df.index[-1].year})')
print(f'Val:   {len(val_df)} samples ({val_df.index[0].year})')
print(f'Test:  {len(test_df)} samples ({test_df.index[0].year})')

In [ ]:
# Scale features
scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_df.values)
val_scaled = scaler.transform(val_df.values)
test_scaled = scaler.transform(test_df.values)

# Target scaler (for inverse transform of predictions)
target_mean = scaler.mean_[target_idx]
target_std = scaler.scale_[target_idx]

print(f'Feature means (train): {scaler.mean_[:3].round(2)}...')
print(f'Feature stds (train):  {scaler.scale_[:3].round(2)}...')
print(f'Target (T degC) — mean: {target_mean:.2f}, std: {target_std:.2f}')

In [ ]:
INPUT_WINDOW = 120    # Past 120 hours (5 days)
FORECAST_HORIZON = 24  # Predict 24 hours ahead
N_FEATURES = train_scaled.shape[1]  # 14

def create_multivariate_sequences(data, input_window, forecast_horizon, target_idx):
    """Create sequences for multivariate-input, univariate-output forecasting.
    
    Args:
        data: scaled numpy array (n_samples, n_features)
        input_window: number of past timesteps as input
        forecast_horizon: how many steps ahead to predict
        target_idx: column index of target variable
    
    Returns:
        X: (n_windows, input_window, n_features)
        y: (n_windows, 1)
    """
    X, y = [], []
    for i in range(len(data) - input_window - forecast_horizon + 1):
        X.append(data[i:i+input_window])  # All features
        y.append(data[i+input_window+forecast_horizon-1, target_idx])  # Target at +horizon
    return torch.FloatTensor(np.array(X)), torch.FloatTensor(np.array(y)).unsqueeze(1)


class ClimateDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# Create sequences
X_train, y_train = create_multivariate_sequences(train_scaled, INPUT_WINDOW, FORECAST_HORIZON, target_idx)
X_val, y_val = create_multivariate_sequences(val_scaled, INPUT_WINDOW, FORECAST_HORIZON, target_idx)
X_test, y_test = create_multivariate_sequences(test_scaled, INPUT_WINDOW, FORECAST_HORIZON, target_idx)

print(f'Training:   X={X_train.shape}, y={y_train.shape}')
print(f'Validation: X={X_val.shape}, y={y_val.shape}')
print(f'Test:       X={X_test.shape}, y={y_test.shape}')

# DataLoaders
BATCH_SIZE = 64
train_loader = DataLoader(ClimateDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(ClimateDataset(X_val, y_val), batch_size=BATCH_SIZE)
test_loader = DataLoader(ClimateDataset(X_test, y_test), batch_size=BATCH_SIZE)

---
## 4. Model Architecture

### Architecture
```
Input: (batch, 120, 14)   ← 5 days × 14 features
       |
  [RNN / LSTM / GRU]      ← processes 120 timesteps
       |
  h_T (last hidden)       ← (batch, hidden_size)
       |
  [Linear Layer]           ← hidden_size → 1
       |
Output: (batch, 1)        ← predicted temperature at +24h
```

In [ ]:
class ClimateForecaster(nn.Module):
    """Multivariate time series forecaster using RNN/LSTM/GRU."""
    
    SUPPORTED = {'RNN': nn.RNN, 'LSTM': nn.LSTM, 'GRU': nn.GRU}
    
    def __init__(self, model_type, input_size, hidden_size, output_size=1,
                 num_layers=1, dropout=0.0, bidirectional=False):
        super().__init__()
        self.model_type = model_type
        self.hidden_size = hidden_size
        
        rnn_cls = self.SUPPORTED[model_type]
        self.rnn = rnn_cls(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional
        )
        
        fc_in = hidden_size * (2 if bidirectional else 1)
        self.fc = nn.Linear(fc_in, output_size)
    
    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out[:, -1, :])
    
    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

In [ ]:
# Parameter comparison
print('Parameter counts (hidden=128, layers=2, input=14):')
for mt in ['RNN', 'LSTM', 'GRU']:
    m = ClimateForecaster(mt, input_size=14, hidden_size=128, num_layers=2, dropout=0.1)
    print(f'  {mt:5s}: {m.count_parameters():>8,} parameters')

---
## 5. Training Infrastructure

In [ ]:
def train_model(model, train_loader, X_val, y_val, epochs, lr,
                device=DEVICE, patience=8, clip_grad=1.0, verbose=True):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
    
    history = {'train_loss': [], 'val_loss': []}
    best_val_loss = float('inf')
    best_state = None
    patience_counter = 0
    
    X_val_d = X_val.to(device)
    y_val_d = y_val.to(device)
    
    start = time.time()
    
    for epoch in range(epochs):
        model.train()
        losses = []
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_b), y_b)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
            optimizer.step()
            losses.append(loss.item())
        
        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_val_d), y_val_d).item()
        
        avg_train = np.mean(losses)
        history['train_loss'].append(avg_train)
        history['val_loss'].append(val_loss)
        scheduler.step(val_loss)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                if verbose: print(f'  Early stopping at epoch {epoch+1}')
                break
        
        if verbose and (epoch + 1) % 5 == 0:
            print(f'  Epoch {epoch+1:3d}/{epochs} | Train: {avg_train:.6f} | Val: {val_loss:.6f}')
    
    elapsed = time.time() - start
    if best_state:
        model.load_state_dict(best_state)
        model = model.to(device)
    return history, elapsed


def evaluate_forecast(model, X_test, y_test, target_mean, target_std, device=DEVICE):
    model.eval()
    with torch.no_grad():
        preds_scaled = model(X_test.to(device)).cpu().numpy()
    
    # Inverse scale
    preds = preds_scaled * target_std + target_mean
    actuals = y_test.numpy() * target_std + target_mean
    
    mse = mean_squared_error(actuals, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(actuals, preds)
    r2 = r2_score(actuals, preds)
    
    return {'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'R2': r2}, preds.flatten(), actuals.flatten()

---
## 6. Hyperparameter Tuning

Using random search with a reduced training budget (10 epochs per trial, 30% data subsample).

In [ ]:
# Subsample training data for faster search
subsample_idx = np.random.choice(len(X_train), size=len(X_train)//3, replace=False)
X_train_sub = X_train[subsample_idx]
y_train_sub = y_train[subsample_idx]
sub_loader = DataLoader(ClimateDataset(X_train_sub, y_train_sub), batch_size=BATCH_SIZE, shuffle=True)

# Search space
SEARCH_SPACE = {
    'hidden_size': [64, 128, 256],
    'num_layers': [1, 2],
    'lr': [1e-4, 5e-4, 1e-3, 5e-3],
    'dropout': [0.0, 0.1, 0.2],
}

N_TRIALS = 10  # per model type
TUNING_EPOCHS = 10

tuning_results = []

for model_type in ['RNN', 'LSTM', 'GRU']:
    print(f'\n{"="*50}')
    print(f'Tuning {model_type} ({N_TRIALS} trials)')
    print(f'{"="*50}')
    
    for trial in range(N_TRIALS):
        config = {k: random.choice(v) for k, v in SEARCH_SPACE.items()}
        
        model = ClimateForecaster(
            model_type=model_type,
            input_size=N_FEATURES,
            hidden_size=config['hidden_size'],
            num_layers=config['num_layers'],
            dropout=config['dropout']
        )
        
        history, t = train_model(model, sub_loader, X_val, y_val,
                                  epochs=TUNING_EPOCHS, lr=config['lr'], verbose=False)
        best_val = min(history['val_loss'])
        
        tuning_results.append({
            'model_type': model_type, 'trial': trial, **config,
            'best_val_loss': best_val, 'params': model.count_parameters()
        })
        print(f'  Trial {trial+1:2d}/{N_TRIALS}: h={config["hidden_size"]:3d}, '
              f'L={config["num_layers"]}, lr={config["lr"]:.4f}, '
              f'd={config["dropout"]:.1f} -> val={best_val:.6f}')

tuning_df = pd.DataFrame(tuning_results)

In [ ]:
# Best configs
best_configs = {}
print('\nBest configurations:')
for mt in ['RNN', 'LSTM', 'GRU']:
    subset = tuning_df[tuning_df['model_type'] == mt]
    best = subset.loc[subset['best_val_loss'].idxmin()]
    best_configs[mt] = best.to_dict()
    print(f'  {mt}: hidden={int(best["hidden_size"])}, layers={int(best["num_layers"])}, '
          f'lr={best["lr"]}, dropout={best["dropout"]}, val_loss={best["best_val_loss"]:.6f}')

---
## 7. Final Training & Comparison

In [ ]:
FINAL_EPOCHS = 30
final_models = {}
final_histories = {}
final_times = {}

for mt in ['RNN', 'LSTM', 'GRU']:
    print(f'\nTraining final {mt}...')
    cfg = best_configs[mt]
    
    model = ClimateForecaster(
        model_type=mt,
        input_size=N_FEATURES,
        hidden_size=int(cfg['hidden_size']),
        num_layers=int(cfg['num_layers']),
        dropout=cfg['dropout']
    )
    
    history, elapsed = train_model(
        model, train_loader, X_val, y_val,
        epochs=FINAL_EPOCHS, lr=cfg['lr']
    )
    
    final_models[mt] = model
    final_histories[mt] = history
    final_times[mt] = elapsed
    print(f'  {mt} done: {len(history["val_loss"])} epochs in {elapsed:.1f}s')

In [ ]:
# Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

for mt in ['RNN', 'LSTM', 'GRU']:
    h = final_histories[mt]
    ax1.plot(h['train_loss'], label=mt, color=COLORS[mt], linewidth=1.5)
    ax2.plot(h['val_loss'], label=mt, color=COLORS[mt], linewidth=1.5)

ax1.set_title('Training Loss'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('MSE'); ax1.legend()
ax2.set_title('Validation Loss'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('MSE'); ax2.legend()
plt.suptitle('Training Curves — Jena Climate', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate on test set
all_metrics = {}
all_preds = {}

for mt in ['RNN', 'LSTM', 'GRU']:
    metrics, preds, actuals = evaluate_forecast(
        final_models[mt], X_test, y_test, target_mean, target_std
    )
    metrics['Time (s)'] = f'{final_times[mt]:.1f}'
    metrics['Params'] = final_models[mt].count_parameters()
    all_metrics[mt] = metrics
    all_preds[mt] = preds

# Persistence baseline: predict current temperature as 24h-ahead forecast
# (i.e., temperature now ≈ temperature in 24h)
persistence_preds = (X_test[:, -1, target_idx].numpy() * target_std + target_mean)
persistence_mse = mean_squared_error(actuals, persistence_preds)
all_metrics['Persistence'] = {
    'MSE': persistence_mse, 'RMSE': np.sqrt(persistence_mse),
    'MAE': mean_absolute_error(actuals, persistence_preds),
    'R2': r2_score(actuals, persistence_preds),
    'Time (s)': '0', 'Params': 0
}

metrics_df = pd.DataFrame(all_metrics).T
for col in ['MSE', 'RMSE', 'MAE', 'R2']:
    metrics_df[col] = metrics_df[col].apply(lambda x: f'{x:.4f}' if isinstance(x, float) else x)

print('\n' + '='*80)
print('TEST SET COMPARISON — Temperature Prediction 24h Ahead')
print('='*80)
metrics_df

In [ ]:
# Predictions plot (2-week window)
plot_len = 24 * 14  # 14 days of hourly data
x_axis = range(plot_len)

fig, ax = plt.subplots(figsize=(18, 6))
ax.plot(x_axis, actuals[:plot_len], color='black', linewidth=1.5, label='Actual', alpha=0.9)
for mt in ['RNN', 'LSTM', 'GRU']:
    ax.plot(x_axis, all_preds[mt][:plot_len], color=COLORS[mt], linewidth=1, 
            label=mt, alpha=0.7, linestyle='--')

ax.set_title('Temperature Predictions — First 2 Weeks of Test Set (2016)', fontsize=14)
ax.set_xlabel('Hours')
ax.set_ylabel('Temperature (°C)')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot: Predicted vs Actual
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, mt in zip(axes, ['RNN', 'LSTM', 'GRU']):
    ax.scatter(actuals, all_preds[mt], alpha=0.1, s=5, color=COLORS[mt])
    ax.plot([actuals.min(), actuals.max()], [actuals.min(), actuals.max()],
            'k--', linewidth=1, label='Perfect prediction')
    ax.set_title(f'{mt} — Predicted vs Actual', fontsize=13)
    ax.set_xlabel('Actual T (°C)')
    ax.set_ylabel('Predicted T (°C)')
    ax.legend()

plt.suptitle('Scatter: Predicted vs Actual Temperature', fontsize=14)
plt.tight_layout()
plt.show()

---
## 8. Analysis & Insights

In [ ]:
# Multi-horizon comparison
print('Multi-Horizon Forecasting Comparison (LSTM only)')
print('='*60)

horizons = [6, 12, 24, 48, 72]
horizon_results = []

for h in horizons:
    X_h, y_h = create_multivariate_sequences(test_scaled, INPUT_WINDOW, h, target_idx)
    if len(X_h) == 0:
        continue
    
    for mt in ['RNN', 'LSTM', 'GRU']:
        metrics, _, _ = evaluate_forecast(final_models[mt], X_h, y_h, target_mean, target_std)
        horizon_results.append({'horizon_hours': h, 'model': mt, 'RMSE': metrics['RMSE'], 'MAE': metrics['MAE']})
    print(f'  Horizon {h:2d}h done')

horizon_df = pd.DataFrame(horizon_results)

In [ ]:
# Plot multi-horizon results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for mt in ['RNN', 'LSTM', 'GRU']:
    sub = horizon_df[horizon_df['model'] == mt]
    ax1.plot(sub['horizon_hours'], sub['RMSE'], 'o-', color=COLORS[mt], label=mt, linewidth=2, markersize=8)
    ax2.plot(sub['horizon_hours'], sub['MAE'], 'o-', color=COLORS[mt], label=mt, linewidth=2, markersize=8)

ax1.set_title('RMSE vs Forecast Horizon', fontsize=13)
ax1.set_xlabel('Forecast Horizon (hours)')
ax1.set_ylabel('RMSE (°C)')
ax1.legend()

ax2.set_title('MAE vs Forecast Horizon', fontsize=13)
ax2.set_xlabel('Forecast Horizon (hours)')
ax2.set_ylabel('MAE (°C)')
ax2.legend()

plt.suptitle('Error Increases with Forecast Horizon', fontsize=14)
plt.tight_layout()
plt.show()

print('Observation: Error grows with horizon — further predictions are harder.')
print('LSTM/GRU degrade more gracefully than vanilla RNN at longer horizons.')

In [ ]:
# Feature ablation: univariate (temp only) vs multivariate (all 14)
print('Feature Ablation: Univariate vs Multivariate')
print('='*50)

# Univariate: only temperature column
train_uni = train_scaled[:, target_idx:target_idx+1]
val_uni = val_scaled[:, target_idx:target_idx+1]
test_uni = test_scaled[:, target_idx:target_idx+1]

X_train_uni, y_train_uni = create_multivariate_sequences(train_uni, INPUT_WINDOW, FORECAST_HORIZON, 0)
X_val_uni, y_val_uni = create_multivariate_sequences(val_uni, INPUT_WINDOW, FORECAST_HORIZON, 0)
X_test_uni, y_test_uni = create_multivariate_sequences(test_uni, INPUT_WINDOW, FORECAST_HORIZON, 0)

uni_loader = DataLoader(ClimateDataset(X_train_uni, y_train_uni), batch_size=BATCH_SIZE, shuffle=True)

ablation_results = []

for mt in ['RNN', 'LSTM', 'GRU']:
    # Univariate model
    model_uni = ClimateForecaster(mt, input_size=1, hidden_size=128, num_layers=1)
    hist_uni, _ = train_model(model_uni, uni_loader, X_val_uni, y_val_uni,
                               epochs=20, lr=0.001, verbose=False)
    metrics_uni, _, _ = evaluate_forecast(model_uni, X_test_uni, y_test_uni, target_mean, target_std)
    
    # Multivariate (already trained)
    metrics_multi, _, _ = evaluate_forecast(final_models[mt], X_test, y_test, target_mean, target_std)
    
    ablation_results.append({
        'Model': mt,
        'Univariate RMSE': metrics_uni['RMSE'],
        'Multivariate RMSE': metrics_multi['RMSE'],
        'Improvement': f"{(1 - metrics_multi['RMSE']/metrics_uni['RMSE'])*100:.1f}%"
    })
    print(f'  {mt}: Uni RMSE={metrics_uni["RMSE"]:.3f}, Multi RMSE={metrics_multi["RMSE"]:.3f}')

ablation_df = pd.DataFrame(ablation_results)
ablation_df

In [ ]:
# Ablation bar chart
x = np.arange(3)
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, ablation_df['Univariate RMSE'], width, label='Univariate (Temp only)', color='#95a5a6')
bars2 = ax.bar(x + width/2, ablation_df['Multivariate RMSE'], width, label='Multivariate (All 14 features)',
               color=[COLORS[m] for m in ['RNN', 'LSTM', 'GRU']])

ax.set_xticks(x)
ax.set_xticklabels(['RNN', 'LSTM', 'GRU'])
ax.set_ylabel('RMSE (°C)')
ax.set_title('Feature Ablation: Univariate vs Multivariate Input', fontsize=14)
ax.legend()

# Add improvement labels
for i, row in ablation_df.iterrows():
    ax.text(i + width/2, row['Multivariate RMSE'] + 0.05, row['Improvement'],
            ha='center', fontweight='bold', fontsize=10, color='green')

plt.tight_layout()
plt.show()

---
## 9. Key Takeaways

### Multivariate Forecasting Insights

1. **Multivariate > Univariate**: Using all 14 weather features improves predictions because correlated variables (humidity, pressure, dew point) provide extra context.

2. **Forecast Horizon Matters**: Error grows with prediction distance. 6h forecasts are much more accurate than 72h forecasts.

3. **LSTM/GRU vs RNN**: With 120-step input windows, gated architectures show clear advantages over vanilla RNN in capturing long-range weather patterns.

4. **Persistence Baseline**: Simply predicting "current temperature" for 24h ahead is surprisingly competitive — always compare against it!

5. **GRU Efficiency**: GRU often matches LSTM accuracy with ~75% of the parameters and faster training.

### Extensions
- Try encoder-decoder for multi-step output (predict next 24 hours simultaneously)
- Add attention mechanism to focus on relevant past timesteps
- Experiment with seasonal features (hour of day, day of year as extra inputs)
- Compare with Prophet or N-BEATS

In [ ]:
# Final summary bar charts
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

models = ['RNN', 'LSTM', 'GRU', 'Persistence']
bar_colors = [COLORS['RNN'], COLORS['LSTM'], COLORS['GRU'], '#95a5a6']

# RMSE
rmse_vals = [float(all_metrics[m]['RMSE']) for m in models]
axes[0].bar(models, rmse_vals, color=bar_colors, edgecolor='white')
axes[0].set_title('RMSE (°C)', fontsize=13)
for bar, v in zip(axes[0].patches, rmse_vals):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05, f'{v:.2f}', ha='center', fontweight='bold')

# Training time
times = [final_times[m] for m in ['RNN', 'LSTM', 'GRU']]
axes[1].bar(['RNN', 'LSTM', 'GRU'], times, color=[COLORS[m] for m in ['RNN', 'LSTM', 'GRU']], edgecolor='white')
axes[1].set_title('Training Time (s)', fontsize=13)

# Parameters
params = [all_metrics[m]['Params'] for m in ['RNN', 'LSTM', 'GRU']]
axes[2].bar(['RNN', 'LSTM', 'GRU'], params, color=[COLORS[m] for m in ['RNN', 'LSTM', 'GRU']], edgecolor='white')
axes[2].set_title('Parameters', fontsize=13)

plt.suptitle('Final Summary — Jena Climate Forecasting', fontsize=14)
plt.tight_layout()
plt.show()

print('Notebook complete! Proceed to Notebook 03 for financial time series.')